<a href="https://colab.research.google.com/github/DiegoiDavid/Gemelo-Digital-Data-Engineering/blob/main/Gold_vG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!python Medallionv3/pipeline_medallion_bbva_v3.py

[0.027s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.027s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/03 21:35:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where

In [ ]:
!pip install google-genai --quiet

In [ ]:
from google import genai
from google.colab import userdata

client_gemini = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

In [ ]:
import glob, json

rutas_manifest = glob.glob(
    "Medallionv3/data/gold/t_mx_master_financial_health/fecha_ejecucion=*/_manifest.json"
)
ruta_manifest_reciente = sorted(rutas_manifest)[-1]

with open(ruta_manifest_reciente, "r", encoding="utf-8") as f:
    manifest = json.load(f)

RUTA_PARQUET_GOLD = f"{manifest['ruta']}/{manifest['archivos_parquet'][0]}"
print(f"📄 Archivo Gold localizado: {RUTA_PARQUET_GOLD}")

📄 Archivo Gold localizado: Medallionv3/data/gold/t_mx_master_financial_health/fecha_ejecucion=2026-10-02/part-00000-7b028f06-e610-4650-bd5b-fe827a65044e-c000.snappy.parquet


In [ ]:
from google.genai import types

ESQUEMA_GOLD = """
Tabla: gold
Columnas:
- id_usuario (string): identificador del usuario, ej. 'USR0001'
- mes (string): mes en formato 'yyyy-MM', ej. '2025-06'
- total_ingresos (double)
- total_egresos (double)
- capacidad_ahorro (double)
- ratio_endeudamiento (double): porcentaje ya expresado como número completo,
  NO como fracción. Ejemplo: el valor 100.0 significa 100% (no 1.0 ni 1%).
  Para "arriba del X%", filtra con ratio_endeudamiento > X (no X/100).
"""

def generar_sql(pregunta_usuario):
    respuesta = client_gemini.models.generate_content(
        model="gemini-3.5-flash",
        contents=pregunta_usuario,
        config=types.GenerateContentConfig(
            system_instruction=(
                "Eres un experto en SQL para DuckDB. Dado el esquema de una tabla "
                "y una pregunta en español, responde ÚNICAMENTE con la consulta SQL "
                "que responde la pregunta. No agregues explicaciones ni marcadores de código. "
                "Solo genera SELECT, nunca INSERT/UPDATE/DELETE/DROP."
                f"\n\nEsquema disponible:\n{ESQUEMA_GOLD}"
            )
        )
    )
    return respuesta.text.strip()

In [ ]:
import duckdb

def ejecutar_sql(consulta_sql):
    if not consulta_sql.strip().upper().startswith("SELECT"):
        raise ValueError(f"⚠️ Consulta rechazada por seguridad (no es SELECT): {consulta_sql}")
    consulta_final = consulta_sql.replace("gold", f"read_parquet('{RUTA_PARQUET_GOLD}')")
    return duckdb.sql(consulta_final).df()

In [ ]:
import time

def preguntar(pregunta_usuario, intentos=3):
    for intento in range(1, intentos + 1):
        try:
            sql = generar_sql(pregunta_usuario)
            print(f"📝 SQL generado:\n{sql}\n")
            return ejecutar_sql(sql)
        except Exception as e:
            if intento == intentos:
                print(f"❌ Falló después de {intentos} intentos: {e}")
                raise
            print(f"⚠️ Intento {intento} falló ({type(e).__name__}), reintentando en 3s...")
            time.sleep(3)

In [ ]:
preguntar("¿cuántos usuarios tienen ratio de endeudamiento arriba de 100%?")

📝 SQL generado:
SELECT COUNT(DISTINCT id_usuario) FROM gold WHERE ratio_endeudamiento > 100



,count(DISTINCT id_usuario)
0,12


In [ ]:
preguntar("¿Cuántos usuarios-mes tuvieron ingresos en 0?")

📝 SQL generado:
SELECT COUNT(*) FROM gold WHERE total_ingresos = 0



,count_star()
0,231


In [ ]:
preguntar("¿Quiénes son los 5 usuarios con mayor capacidad de ahorro?")

⚠️ Intento 1 falló (ServerError), reintentando en 3s...
⚠️ Intento 2 falló (ServerError), reintentando en 3s...
📝 SQL generado:
SELECT id_usuario, SUM(capacidad_ahorro) AS total_capacidad
FROM gold
GROUP BY id_usuario
ORDER BY total_capacidad DESC
LIMIT 5



,id_usuario,total_capacidad
0,USR0027,60544.96
1,USR0023,24047.77
2,USR0010,22043.10
3,USR0004,20412.05
4,USR0006,12562.25
